# GOLD (Dados Agregados)
Métricas e agregações<br>
Dados prontos para análise<br>
Otimizados para consulta<br>

## PROCESSAMENTO DOS DADOS

### IMPORTAÇÃO DAS BIBLIOTECAS

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime
import seaborn as sns
import numpy as np

### CARREGAMENTO DOS DADOS

In [2]:
gold_path = 'data/gold/dados_tratados (1).csv'
df = pd.read_csv(gold_path)

## CRIAÇÃO DE NOVAS FEATURES

In [3]:
idade_faixas = [0, 20, 30, 45, 55, 100]
idade_categoria = ["Até 20", "21 a 30", "31 a 45", "46 a 55", "Maior que 55"]

df["FAIXA_ETARIA"] = pd.cut(df["IDADE"], idade_faixas, labels=idade_categoria)
df["FAIXA_ETARIA"].value_counts()

,count
FAIXA_ETARIA,
21 a 30,2994
46 a 55,2664
31 a 45,2568
Maior que 55,1692
Até 20,558


In [4]:
df['TEM_FILHOS'] = np.where(df['QT_FILHOS'] > 0, 'Sim', 'Não')
df["TEM_FILHOS"].value_counts()

,count
TEM_FILHOS,
Sim,7147
Não,3329


In [5]:
df['RENDA_TOTAL'] = df['ULTIMO_SALARIO'] + df['OUTRA_RENDA_VALOR']
df['RENDA_TOTAL'].head()

,RENDA_TOTAL
0,1800.0
1,4800.0
2,2200.0
3,3900.0
4,6100.0


In [6]:
df.groupby(['RENDA_TOTAL']).size()

,0
RENDA_TOTAL,
1800.0,846
2200.0,792
3100.0,792
3900.0,792
4500.0,468
4800.0,792
6100.0,524
8500.0,522
9000.0,522


In [7]:
df['CATEGORIA_RENDA'] = pd.cut(
    df['RENDA_TOTAL'],
    bins=[0, 2500, 5000, 10000, 20000, float('inf')],
    labels=['Baixa', 'Média-Baixa', 'Média', 'Média-Alta', 'Alta']
)
df['CATEGORIA_RENDA'].value_counts()

,count
CATEGORIA_RENDA,
Média-Alta,2878
Média-Baixa,2844
Média,2648
Baixa,1638
Alta,468


In [8]:

col_bool = ['TEM_FILHOS', 'TRABALHANDO_ATUALMENTE', 'CASA_PROPRIA']

for col in col_bool:
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper().map({'SIM': 1, 'NÃO': 0, 'NAO': 0})
        df[col] = df[col].fillna(0)

# ===============================
# AGREGAÇÃO 1: Análise de Métricas por Estados
# ===============================

metricas_estado = df.groupby('UF').agg({
    'CODIGO_CLIENTE': 'count',
    'RENDA_TOTAL': 'mean',
    'SCORE': 'mean',
    'QT_IMOVEIS': 'mean',
    'QT_CARROS': 'mean',
    'TEM_FILHOS': 'mean'
}).reset_index()

metricas_estado.columns = [
    'UF', 'total_clientes', 'renda_media', 'score_medio',
    'media_imoveis', 'media_carros', 'percentual_com_filhos'
]

metricas_estado['percentual_com_filhos'] *= 100
metricas_estado.to_csv('data/gold/metricas_estado.csv', index=False)

In [9]:
# ===============================
# AGREGAÇÃO 2: Análise de Clientes
# ===============================

analise_clientes = df[[
    'CODIGO_CLIENTE', 'IDADE', 'FAIXA_ETARIA', 'RENDA_TOTAL',
    'CATEGORIA_RENDA', 'SCORE', 'QT_IMOVEIS', 'QT_CARROS',
    'ULTIMO_SALARIO', 'TRABALHANDO_ATUALMENTE'
]]

analise_clientes['capacidade_credito'] = (
    analise_clientes['RENDA_TOTAL'].fillna(0) * 0.3 + analise_clientes['SCORE'].fillna(0) * 10
)

analise_clientes.to_csv('data/gold/analise_clientes.csv', index=False)

/tmp/ipython-input-1996562029.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  analise_clientes['capacidade_credito'] = (


In [10]:
# ===============================
# AGREGAÇÃO 3: Análise de Patrimônio
# ===============================

ativos = df.groupby('FAIXA_ETARIA', observed=False).agg({
    'QT_IMOVEIS': 'mean',
    'VL_IMOVEIS': 'mean',
    'QT_CARROS': 'mean',
    'VALOR_TABELA_CARROS': 'mean',
    'RENDA_TOTAL': 'mean',
    'SCORE': 'mean'
}).reset_index()

ativos.columns = [
    'faixa_etaria', 'media_qt_imoveis', 'media_valor_imoveis',
    'media_qt_carros', 'media_valor_carros', 'renda_media', 'score_medio'
]

ativos.to_csv('data/gold/ativos_patrimonio.csv', index=False)

In [11]:
df.head()

,CODIGO_CLIENTE,UF,IDADE,ESCOLARIDADE,ESTADO_CIVIL,QT_FILHOS,CASA_PROPRIA,QT_IMOVEIS,VL_IMOVEIS,OUTRA_RENDA,...,TEMPO_ULTIMO_EMPREGO_MESES,TRABALHANDO_ATUALMENTE,ULTIMO_SALARIO,QT_CARROS,VALOR_TABELA_CARROS,SCORE,FAIXA_ETARIA,TEM_FILHOS,RENDA_TOTAL,CATEGORIA_RENDA
0,1,4,19,2,2,0,0.0,0,0.0,0,...,8,0.0,1800.0,0,0.0,12.000000,Até 20,0,1800.0,Baixa
1,2,0,23,1,2,1,0.0,0,0.0,0,...,9,0.0,4800.0,1,50000.0,18.000000,21 a 30,1,4800.0,Média-Baixa
2,3,3,25,0,0,0,0.0,1,220000.0,0,...,18,0.0,2200.0,2,30000.0,23.000000,21 a 30,0,2200.0,Baixa
3,4,1,27,2,0,1,0.0,0,0.0,0,...,22,0.0,3900.0,0,0.0,28.666667,21 a 30,1,3900.0,Média-Baixa
4,5,2,28,1,1,2,0.0,1,370000.0,0,...,30,0.0,6100.0,1,35000.0,34.166667,21 a 30,1,6100.0,Média


In [12]:
print("Agregações criadas e salvas na camada Gold")

Agregações criadas e salvas na camada Gold
